# Download Data From Kaggle

For more information about the competition check the [link](https://www.kaggle.com/competitions/titanic/)

In [170]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running on Google Colab. Setting up Kaggle data...")
    from google.colab import userdata
    import json

    # Setup Kaggle API
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

    !mkdir -p ~/.kaggle
    kaggle_config = {
        "username": os.environ["KAGGLE_USERNAME"],
        "key": os.environ["KAGGLE_KEY"],
    }
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_config, f)
    !chmod 600 ~/.kaggle/kaggle.json

    # Download data if it doesn't exist yet
    if not os.path.exists("data"):
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded and unzipped.")
else:
    if not os.path.exists("data"):
        print("Download data ...")
        !kaggle competitions download -c titanic
        !unzip -q titanic.zip -d data/
        !rm titanic.zip
        print("Data downloaded")
    else:
        print("Data already exists")

Running on Google Colab. Setting up Kaggle data...


# Exploring The Dataset

In [171]:
import pandas as pd

df = pd.read_csv("data/train.csv")

### Missing values

### Filling mising values

In [172]:
mean_age = df["Age"].mean()
df["Age"] = df["Age"].fillna(mean_age)
df["Embarked"] = df["Embarked"].fillna("S")
df["Cabin"] = df["Cabin"].ffill()
df.drop(
    ["Embarked", "Cabin", "Ticket", "Name", "PassengerId", "Fare"], axis=1, inplace=True
)

In [173]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Sex"] = le.fit_transform(df["Sex"])

In [174]:
from sklearn.model_selection import train_test_split

Y = df["Survived"]
X = df.drop(["Survived"], axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [192]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from scipy.stats import uniform, randint

# pipeline = Pipeline(
#     [
#         ("scaller", StandardScaler()),
#         ("clf", SVC()),
#     ]
# )

# param_distribution = [
#     {
#         # "scaller": [StandardScaler()],
#         # "clf": [SVC()],
#         "C": uniform(0.1, 2).rvs(150),
#         "kernel": ["poly", "linear", "rbf"],
#         "degree": [2, 3, 4, 5, 6, 7],
#         "class_weight": ["balanced", None],
#         "tol": uniform(0.1, 2).rvs(150),
#         "probability": [True, False],
#         "coef0": uniform(0.9, 1.5).rvs(150),
#     },
# ]
param_distribution = [
    {
        # "scaller": [StandardScaler()],
        # "clf": [SVC()],
        "C": uniform(0.1, 2).rvs(150),
        "kernel": ["poly", "linear", "rbf"],
        "degree": [2, 3, 4, 5, 6, 7],
        "class_weight": ["balanced", None],
        "tol": uniform(0.1, 2).rvs(150),
        "probability": [True, False],
        "coef0": uniform(0.9, 1.5).rvs(150),
    },
]

search = RandomizedSearchCV(
    estimator=LogisticRegression(),
    param_distributions={},
    verbose=3,
    n_iter=10,
    cv=KFold(n_splits=5),
    n_jobs=-1,
)


search.fit(X_train, Y_train)
print(f"best_score: {search.best_score_}")
print(f"best_params: {search.best_params_}")

Fitting 5 folds for each of 1 candidates, totalling 5 fits
best_score: 0.7878656554712892
best_params: {}


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [193]:
pred = search.best_estimator_.predict(X_test)
scores = search.best_estimator_.score(X_test, Y_test)
scores

0.8268156424581006

In [177]:
!ls

data  sample_data  submission.csv


In [190]:
# df_test = pd.read_csv("data/test.csv")
# mean_age = df_test["Age"].mean()
# df_test["Age"] = df_test["Age"].fillna(mean_age)
# df_test["Embarked"] = df_test["Embarked"].fillna("S")
# df_test["Cabin"] = df_test["Cabin"].ffill()
# df_test.shape[0]
# df_test.drop(
#     ["Embarked", "Cabin", "Ticket", "Name", "Fare"], axis=1, inplace=True
# )
# le = LabelEncoder()
# df_test["Sex"] = le.fit_transform(df_test["Sex"])

# preds = search.best_estimator_.predict(df_test.drop("PassengerId", axis=1))
# preds

# output = pd.DataFrame({'PassengerId': df_test["PassengerId"], 'Survived': preds})
# output.to_csv('submission.csv', index=False)

# !kaggle competitions submit -c titanic -f submission.csv  -m "Message"


100% 2.77k/2.77k [00:00<00:00, 3.07kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster